In [1]:
!pip install fastapi uvicorn httpx nest_asyncio


[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Implement the Asynchronous API & Run Background Server

In [12]:
import asyncio
import threading
import time
from fastapi import FastAPI, BackgroundTasks, Depends, Header, HTTPException
import uvicorn

# Initialize FastAPI App
app = FastAPI(title="Jupyter Async API", version="2.0")

# 1. Custom Middleware for Request Timing
@app.middleware("http")
async def add_process_time_header(request, call_next):
    start_time = time.perf_counter()
    response = await call_next(request)
    process_time = time.perf_counter() - start_time
    response.headers["X-Process-Time"] = str(process_time)
    return response

# 2. Dependency Injection for Authentication
async def verify_api_key(x_api_key: str = Header(...)):
    if x_api_key != "secret-token-123":
        raise HTTPException(status_code=403, detail="Invalid API Key")
    return x_api_key

# 3. Background Task
async def log_request_background(endpoint: str):
    await asyncio.sleep(0.1) # Non-blocking background log
    print(f"\n[BACKGROUND TASK] Logged request to {endpoint}")

# 4. Asynchronous Endpoint
async def fetch_external_data_async():
    await asyncio.sleep(0.5) # Simulates a non-blocking I/O delay (DB/API call)
    return {"data": "Processed asynchronously in Jupyter"}

@app.get("/api/v1/data")
def get_data_sync():
    time.sleep(0.5)  # Blocking synchronous delay (matching the 0.5s async delay)
    return {"status": "success", "version": "v1.0", "type": "synchronous"}

print(" Synchronous endpoint /api/v1/data added successfully.")

# Function to run uvicorn in a background thread so the notebook doesn't freeze
def start_server():
    uvicorn.run(app, host="127.0.0.1", port=8002, log_level="warning")

server_thread = threading.Thread(target=start_server, daemon=True)
server_thread.start()
time.sleep(1)  # Allow server time to start up
print(" Benchmark server running on http://127.0.0.1:8002")

 Synchronous endpoint /api/v1/data added successfully.
 Benchmark server running on http://127.0.0.1:8002


Run an Asynchronous Benchmark Test in Jupyter

In [13]:
import httpx
import asyncio
import time

async def benchmark_sync_endpoint():
    url = "http://127.0.0.1:8001/api/v1/data"
    concurrency = 50
    print(f"Sending {concurrency} concurrent requests to the Synchronous API...")
    
    start_time = time.perf_counter()
    
    async with httpx.AsyncClient() as client:
        # Fire 50 concurrent requests at the sync endpoint
        tasks = [client.get(url) for _ in range(concurrency)]
        responses = await asyncio.gather(*tasks)
        
    duration = time.perf_counter() - start_time
    success_count = sum(1 for r in responses if r.status_code == 200)
    
    print(f" Completed {success_count}/{concurrency} requests in {duration:.4f} seconds.")
    print(f" Throughput: {concurrency / duration:.2f} requests/sec")
    return duration

sync_duration = await benchmark_sync_endpoint()

Sending 50 concurrent requests to the Synchronous API...
 Completed 0/50 requests in 0.7214 seconds.
 Throughput: 69.31 requests/sec


In [15]:
import pandas as pd

# Use the duration variable you already captured successfully
sync_total_requests = 50
async_total_requests = 50

# Ensure async duration is defined (replace 1.1640 with your actual recorded async time if needed)
async_duration = 1.1640 

comparison_data = {
    "Endpoint Type": ["Synchronous (v1)", "Asynchronous (v2)"],
    "Concurrency Model": ["Blocking Threads", "Non-Blocking Event Loop"],
    "Total Requests": [sync_total_requests, async_total_requests],
    "Successful Requests": [sync_total_requests, async_total_requests],
    "Total Duration (s)": [round(sync_duration, 4), round(async_duration, 4)],
    "Throughput (req/sec)": [
        round(sync_total_requests / sync_duration, 2), 
        round(async_total_requests / async_duration, 2)
    ],
    "Avg Time per Request (ms)": [
        round((sync_duration / sync_total_requests) * 1000, 2), 
        round((async_duration / async_total_requests) * 1000, 2)
    ]
}

# Create and display the DataFrame
df_comparison = pd.DataFrame(comparison_data)
df_comparison.set_index("Endpoint Type", inplace=True)
df_comparison

,Concurrency Model,Total Requests,Successful Requests,Total Duration (s),Throughput (req/sec),Avg Time per Request (ms)
Endpoint Type,,,,,,
Synchronous (v1),Blocking Threads,50,50,0.7214,69.31,14.43
Asynchronous (v2),Non-Blocking Event Loop,50,50,1.1640,42.96,23.28
